# Step 8: The Transformer

The previous notebook showed that attention solves the bottleneck — but RNNs are still there,
processing sequences one step at a time. This means:
- Training is sequential: step T depends on step T-1
- Long-range attention is routed through many hidden states
- GPUs are underutilized

**"Attention Is All You Need"** (Vaswani et al., 2017) removed the RNNs entirely.
The Transformer processes **all positions in parallel** using **self-attention** —
each position attends directly to every other position in one matrix operation.

```
Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V
```

The denominator √dₖ prevents the dot products from growing large with dimensionality,
which would push softmax into saturation.

**What you'll see:**
1. Self-attention from scratch: Q, K, V projections, scaled dot product
2. Multi-head attention: learn different relationships simultaneously
3. Positional encoding: how position information enters without recurrence
4. A full Transformer encoder block
5. Self-attention patterns: what positions attend to what

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Self-Attention from Scratch

For each position, we compute three vectors by multiplying the input by learned matrices:
- **Query (Q)**: what this position is looking for
- **Key (K)**: what this position contains
- **Value (V)**: what this position will contribute if attended to

Attention weights = softmax(Q·Kᵀ / √dₖ)  — how much each position attends to each other
Output = weights · V                        — weighted sum of values

**Crucially**: this is a single matrix multiply — all positions computed in parallel.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, heads, seq_len, d_k)
    mask:    (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
    Returns: output (batch, heads, seq_len, d_k), weights (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, T, T)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = F.softmax(scores, dim=-1)   # each row sums to 1
    output  = torch.matmul(weights, V)    # (B, H, T, d_k)
    return output, weights


# Demo: 3-word sentence, d_model=4
d_model = 4
T = 3  # sequence length

# Pretend embedding: 3 words, each a 4-dim vector
X = torch.tensor([[1.,0.,0.,0.],  # word 0
                   [0.,1.,0.,0.],  # word 1
                   [0.,0.,1.,0.]]) # word 2
X = X.unsqueeze(0)  # (1, 3, 4)

# Random projection matrices (in practice, these are learned)
Wq = torch.randn(d_model, d_model) * 0.5
Wk = torch.randn(d_model, d_model) * 0.5
Wv = torch.randn(d_model, d_model) * 0.5

Q = X @ Wq  # (1, 3, 4)
K = X @ Wk
V = X @ Wv

# Add head dimension
Q = Q.unsqueeze(1); K = K.unsqueeze(1); V = V.unsqueeze(1)

output, weights = scaled_dot_product_attention(Q, K, V)
print("Input shape:", X.shape)
print("Attention weights (how each word attends to every other):")
print(weights[0, 0].detach().numpy().round(3))
print("Output shape:", output.shape, "(same as input — each position gets a new representation)")

## Multi-Head Attention

One attention head learns one kind of relationship. To learn multiple relationships
(subject-verb, adjective-noun, reference-antecedent), we run attention in parallel
with H different Q/K/V projections, then concatenate and project:

```
MultiHead(Q,K,V) = Concat(head₁, ..., headₕ) · Wₒ
where headᵢ = Attention(Q·Wᵢq, K·Wᵢk, V·Wᵢv)
```

Each head uses d_k = d_model / H, so total compute is the same as one big head
but with richer, multi-relational representations.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k    = d_model // n_heads
        self.n_heads = n_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        H, dk = self.n_heads, self.d_k

        # Project and reshape to (B, H, T, d_k)
        Q = self.Wq(x).view(B, T, H, dk).transpose(1, 2)  # (B, H, T, dk)
        K = self.Wk(x).view(B, T, H, dk).transpose(1, 2)
        V = self.Wv(x).view(B, T, H, dk).transpose(1, 2)

        out, weights = scaled_dot_product_attention(Q, K, V, mask)

        # Merge heads: (B, H, T, dk) → (B, T, D)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.Wo(out), weights  # project merged heads


# Quick verification
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)  # (batch=2, seq=10, d_model=64)
out, w = mha(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Attention weights: {w.shape}  (B=2, H=8, T=10, T=10)")
print(f"Weights sum to 1 along last dim: {w[0,0,0].sum().item():.4f}")

## Positional Encoding

Self-attention is **permutation invariant** — shuffle the input sequence and
the output is the same (but shuffled). This is great for sets, but sequences
have order. We must inject position information.

Vaswani et al. use fixed sinusoidal encodings:
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

These are **added** to the token embeddings. The sinusoids at different frequencies
allow the model to infer both absolute position and relative position between tokens.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Precompute PE matrix: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()        # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2).float() *   # (d_model/2,)
                        (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)  # even indices
        pe[:, 1::2] = torch.cos(pos * div)  # odd indices
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# Visualize positional encodings
d_model = 64
max_len = 50
pe_module = PositionalEncoding(d_model, max_len, dropout=0.0)
pe_matrix = pe_module.pe[0].numpy()  # (50, 64)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap of the PE matrix
im = axes[0].imshow(pe_matrix.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xlabel('Position (token index)')
axes[0].set_ylabel('Embedding dimension')
axes[0].set_title('Sinusoidal positional encodings\n(rows = dimensions, columns = positions)')
plt.colorbar(im, ax=axes[0])

# Show dot-product similarity between position encodings
sim = pe_matrix @ pe_matrix.T  # (50, 50)
im2 = axes[1].imshow(sim, cmap='Blues')
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Position')
axes[1].set_title('Dot product similarity between positions\n(nearby positions are more similar)')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## Full Transformer Encoder Block

One Transformer layer = self-attention + feedforward network, with residual connections
and layer normalization after each sub-layer (post-LN; pre-LN is now preferred but
this matches the original paper):

```
x = LayerNorm(x + MultiHeadAttention(x))   # attention sub-layer
x = LayerNorm(x + FFN(x))                  # feedforward sub-layer

FFN(x) = max(0, xW₁ + b₁)W₂ + b₂  (two linear layers with ReLU; d_ff = 4 × d_model)
```

The **residual connections** allow gradients to flow directly through the network
regardless of depth — same insight as ResNets, but applied to sequence models.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)


class TransformerEncoderBlock(nn.Module):
    """One layer of the Transformer encoder (post-LN, as in original paper)."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention sub-layer with residual connection
        attn_out, weights = self.attn(x, mask)
        x = self.ln1(x + self.drop(attn_out))
        # Feedforward sub-layer with residual connection
        x = self.ln2(x + self.ff(x))
        return x, weights


class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pe    = PositionalEncoding(d_model, max_len, dropout)
        self.scale = math.sqrt(d_model)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, mask=None):
        # Scale embeddings before adding positional encoding
        x = self.pe(self.embed(x) * self.scale)
        all_weights = []
        for layer in self.layers:
            x, w = layer(x, mask)
            all_weights.append(w)
        return x, all_weights


# Build a small encoder
D_MODEL  = 64
N_HEADS  = 4
D_FF     = 256
N_LAYERS = 2
VOCAB    = 100

encoder = TransformerEncoder(VOCAB, D_MODEL, N_HEADS, D_FF, N_LAYERS).to(device)
total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {total_params:,}")
print()

# Verify shapes
x = torch.randint(0, VOCAB, (2, 10)).to(device)  # (batch=2, seq=10)
out, weights = encoder(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}   (same shape — contextual representations)")
print(f"Attention weights from {len(weights)} layers, each {weights[0].shape}")

In [ ]:
# Visualize attention patterns on a small task: classify sequence parity
# Task: given a sequence of 0s and 1s, does it contain more 1s or 0s?

class ParityClassifier(nn.Module):
    def __init__(self, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.encoder = TransformerEncoder(3, d_model, n_heads, d_model*4, n_layers,
                                          max_len=32, dropout=0.0)
        self.head = nn.Linear(d_model, 2)  # binary classification

    def forward(self, x):
        out, weights = self.encoder(x)
        cls = out[:, 0]  # use first position as summary (like BERT's [CLS])
        return self.head(cls), weights


# Generate task: sequences with a prepended [CLS]=2 token
def make_parity_batch(batch_size=64, seq_len=8):
    seqs  = torch.randint(0, 2, (batch_size, seq_len))  # 0s and 1s
    cls   = torch.full((batch_size, 1), 2)
    x     = torch.cat([cls, seqs], dim=1)
    label = (seqs.sum(dim=1) > seq_len // 2).long()  # 1 if more 1s than 0s
    return x, label


clf = ParityClassifier(d_model=32, n_heads=4, n_layers=2).to(device)
optimizer = torch.optim.Adam(clf.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

losses, accs = [], []
for step in range(1000):
    x, y = make_parity_batch(64, 8)
    x, y = x.to(device), y.to(device)
    logits, _ = clf(x)
    loss = criterion(logits, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

    if (step + 1) % 200 == 0:
        acc = (logits.argmax(-1) == y).float().mean().item()
        print(f"Step {step+1} | loss: {loss.item():.4f} | acc: {acc:.3f}")

In [ ]:
# Visualize self-attention patterns on a specific example
clf.eval()
example_seq = [0, 1, 1, 0, 1, 0, 0, 1]  # equal 0s and 1s
x_ex = torch.tensor([[2] + example_seq]).to(device)  # prepend CLS=2

with torch.no_grad():
    logits, all_weights = clf(x_ex)

pred_class = logits.argmax(-1).item()
labels_str = ['[CLS]'] + [str(t) for t in example_seq]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'Self-attention weights — input: [CLS] {example_seq} → prediction: {pred_class}',
             fontsize=12)

for layer_idx, layer_weights in enumerate(all_weights):
    for head_idx in range(4):
        ax = axes[layer_idx, head_idx]
        w = layer_weights[0, head_idx].cpu().numpy()  # (T, T)
        im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
        ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=9)
        ax.set_xticks(range(len(labels_str)))
        ax.set_xticklabels(labels_str, fontsize=7, rotation=45)
        ax.set_yticks(range(len(labels_str)))
        ax.set_yticklabels(labels_str, fontsize=7)

plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Self-attention** | Each position attends to all others — O(T²) operations, fully parallel |
| **Q/K/V** | Query = what I look for; Key = what I have; Value = what I give |
| **Scaled dot product** | Divide by √dₖ to prevent softmax saturation |
| **Multi-head** | H parallel heads learn different relationship types |
| **Positional encoding** | Sinusoids add position info that attention alone ignores |
| **FFN** | Position-wise MLP applies the same transformation at every token independently |
| **Residual + LayerNorm** | Enables training deep stacks; gradient flows cleanly |

**Critical property**: unlike RNNs, any two positions interact directly — the path length
between any two tokens is O(1), not O(T). This enables learning long-range dependencies
that RNNs struggle with.

**Training**: fully parallelizable over the sequence dimension — GPUs can process all
positions simultaneously, unlocking massive training scale.

Next: use a **decoder-only** Transformer for language modeling — this is GPT.